# 0: Installing Prerequisites

In [1]:
# 0: Installing Prerequisites
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score

#0.1 Loading Dataset

In [3]:
print("Loading Dataset...")
df = pd.read_csv("https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv")
print("Dataset for homework 4 loaded successfully!")

Loading Dataset...
Dataset for homework 4 loaded successfully!


Identifying the categorical and numerical columns

In [4]:
print(f"Dataset shape: {df.shape}")

Dataset shape: (1462, 9)


In [6]:
categorical_columns = list(df.columns[df.dtypes == 'object'])
numerical_columns = list(df.columns[df.dtypes != 'object'])
print(f"Categorical columns: {categorical_columns}")
print(f"Numerical columns: {numerical_columns}")

Categorical columns: ['lead_source', 'industry', 'employment_status', 'location']
Numerical columns: ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score', 'converted']


In [7]:
print(df.isnull().sum())

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64


Fill missing values

In [8]:
df[categorical_columns] = df[categorical_columns].fillna('NA')
df[numerical_columns] = df[numerical_columns].fillna(0.0)

In [9]:
print("Missing values after handling:")
print(df.isnull().sum())

Missing values after handling:
lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


##0.3 Split the data (60/20/20) with random_state=1

In [12]:
print("SPLITTING DATA (60/20/20)")
print()

# First split: 80% full_train, 20% test
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

# Second split: 75% train (60% of original), 25% validation (20% of original)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

print(f"Full dataset: {len(df)} ({100:.1f}%)")
print(f"Train set: {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)")
print(f"Validation set: {len(df_val)} ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test set: {len(df_test)} ({len(df_test)/len(df)*100:.1f}%)")

SPLITTING DATA (60/20/20)

Full dataset: 1462 (100.0%)
Train set: 876 (59.9%)
Validation set: 293 (20.0%)
Test set: 293 (20.0%)


In [13]:
# Separate target variable
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# Keep features separate
df_train_features = df_train.drop('converted', axis=1).reset_index(drop=True)
df_val_features = df_val.drop('converted', axis=1).reset_index(drop=True)
df_test_features = df_test.drop('converted', axis=1).reset_index(drop=True)

print("Target variable separated from features.")

Target variable separated from features.


## Question 1: ROC AUC Feature Importance

In [14]:
# Numerical variables to test
numerical_vars = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']

# Get the original training data with target
df_train_with_target = df_train.reset_index(drop=True)

auc_scores = {}

for var in numerical_vars:
    # Use the variable as score (prediction)
    score = df_train_with_target[var].values
    y = df_train_with_target['converted'].values

    # Calculate AUC
    auc = roc_auc_score(y, score)

    # If AUC < 0.5, invert the variable
    if auc < 0.5:
        score = -score
        auc = roc_auc_score(y, score)
        print(f"{var}: AUC = {auc:.6f} (inverted)")
    else:
        print(f"{var}: AUC = {auc:.6f}")

    auc_scores[var] = auc

print()
max_auc_var = max(auc_scores, key=auc_scores.get)
print(f"Variable with highest AUC: {max_auc_var}")
print(f"Highest AUC: {auc_scores[max_auc_var]:.6f}")
print()
print(f"Answer Q1: {max_auc_var}")

lead_score: AUC = 0.614499
number_of_courses_viewed: AUC = 0.763568
interaction_count: AUC = 0.738270
annual_income: AUC = 0.551958

Variable with highest AUC: number_of_courses_viewed
Highest AUC: 0.763568

Answer Q1: number_of_courses_viewed


##Question 2: Training the Model - AUC

In [15]:
# Prepare data with one-hot encoding using DictVectorizer
train_dict = df_train_features.to_dict(orient='records')
val_dict = df_val_features.to_dict(orient='records')

# Initialize and fit DictVectorizer
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")
print()

# Train Logistic Regression model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

# Predict probabilities
y_pred_proba = model.predict_proba(X_val)[:, 1]

# Calculate AUC
auc = roc_auc_score(y_val, y_pred_proba)
auc_rounded = round(auc, 3)

print(f"Validation AUC: {auc:.6f}")
print(f"Validation AUC (rounded to 3 decimals): {auc_rounded}")
print()
print(f"Answer Q2: {auc_rounded}")

Training features shape: (876, 31)
Validation features shape: (293, 31)

Validation AUC: 0.817132
Validation AUC (rounded to 3 decimals): 0.817

Answer Q2: 0.817


## Question 3: Precision and Recall Intersection

In [23]:
# Evaluate at thresholds from 0.0 to 1.0 with step 0.01
thresholds = np.arange(0.0, 1.01, 0.01)

precisions = []
recalls = []

for threshold in thresholds:
    # Make predictions based on threshold
    y_pred = (y_pred_proba >= threshold).astype(int)

    # Calculate precision and recall
    if y_pred.sum() == 0:
        precision = 0
    else:
        precision = precision_score(y_val, y_pred, zero_division=0)

    recall = recall_score(y_val, y_pred, zero_division=0)

    precisions.append(precision)
    recalls.append(recall)

# Convert to arrays
precisions = np.array(precisions)
recalls = np.array(recalls)

# Find intersection point (where they are closest and both > 0)
valid_indices = (precisions > 0) & (recalls > 0)
valid_thresholds = thresholds[valid_indices]
valid_precisions = precisions[valid_indices]
valid_recalls = recalls[valid_indices]
valid_differences = np.abs(valid_precisions - valid_recalls)

min_diff_idx = np.argmin(valid_differences)
intersection_threshold = valid_thresholds[min_diff_idx]

print(f"Precision and recall curves intersect at threshold: {intersection_threshold:.3f}")
print(f"Precision at intersection: {valid_precisions[min_diff_idx]:.6f}")
print(f"Recall at intersection: {valid_recalls[min_diff_idx]:.6f}")
print()

# Check closest option (0.145, 0.345, 0.545, 0.745)
option_thresholds_q3 = [0.145, 0.345, 0.545, 0.745]
distances = [abs(opt - intersection_threshold) for opt in option_thresholds_q3]
closest_option = option_thresholds_q3[np.argmin(distances)]

print(f"Answer Q3: {closest_option:.3f}")

Precision and recall curves intersect at threshold: 0.640
Precision at intersection: 0.779070
Recall at intersection: 0.783626

Answer Q3: 0.545


## Question 4: F1 Score

In [17]:
# Calculate F1 score for all thresholds
f1_scores = []

for i, threshold in enumerate(thresholds):
    P = precisions[i]
    R = recalls[i]

    # Calculate F1 score: F1 = 2 * (P * R) / (P + R)
    if P + R == 0:
        f1 = 0
    else:
        f1 = 2 * (P * R) / (P + R)

    f1_scores.append(f1)

# Convert to array
f1_scores = np.array(f1_scores)

# Find threshold with maximum F1
max_f1_idx = np.argmax(f1_scores)
max_f1_threshold = thresholds[max_f1_idx]
max_f1_value = f1_scores[max_f1_idx]

print(f"Maximum F1 score: {max_f1_value:.6f}")
print(f"Occurs at threshold: {max_f1_threshold:.3f}")
print()

# Check closest option (0.14, 0.34, 0.54, 0.74)
option_thresholds_q4 = [0.14, 0.34, 0.54, 0.74]
distances = [abs(opt - max_f1_threshold) for opt in option_thresholds_q4]
closest_option = option_thresholds_q4[np.argmin(distances)]

print(f"Closest option: {closest_option:.2f}")
print()
print(f"Answer Q4: {closest_option:.2f}")

Maximum F1 score: 0.812500
Occurs at threshold: 0.570

Closest option: 0.54

Answer Q4: 0.54


## Question 5: 5-Fold Cross-Validation

In [18]:
# Use KFold on df_full_train
kfold = KFold(n_splits=5, shuffle=True, random_state=1)

fold_scores = []

# Iterate over folds
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(df_full_train)):
    # Split the data
    df_fold_train = df_full_train.iloc[train_idx]
    df_fold_val = df_full_train.iloc[val_idx]

    # Separate features and target
    y_fold_train = df_fold_train['converted'].values
    y_fold_val = df_fold_val['converted'].values

    X_fold_train_df = df_fold_train.drop('converted', axis=1)
    X_fold_val_df = df_fold_val.drop('converted', axis=1)

    # One-hot encode
    train_dict_fold = X_fold_train_df.to_dict(orient='records')
    val_dict_fold = X_fold_val_df.to_dict(orient='records')

    dv_fold = DictVectorizer(sparse=False)
    X_fold_train = dv_fold.fit_transform(train_dict_fold)
    X_fold_val = dv_fold.transform(val_dict_fold)

    # Train model
    model_fold = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
    model_fold.fit(X_fold_train, y_fold_train)

    # Predict and calculate AUC
    y_fold_pred_proba = model_fold.predict_proba(X_fold_val)[:, 1]
    auc_fold = roc_auc_score(y_fold_val, y_fold_pred_proba)

    fold_scores.append(auc_fold)
    print(f"Fold {fold_idx + 1}: AUC = {auc_fold:.6f}")

# Calculate mean and std
fold_scores = np.array(fold_scores)
mean_auc = np.mean(fold_scores)
std_auc = np.std(fold_scores)

print()
print(f"Mean AUC: {mean_auc:.6f}")
print(f"Standard Deviation: {std_auc:.6f}")
print()

# Round to 3 decimals
std_auc_rounded = round(std_auc, 3)
print(f"Standard Deviation (rounded to 3 decimals): {std_auc_rounded}")
print()
print(f"Answer Q5: {std_auc_rounded}")

Fold 1: AUC = 0.806075
Fold 2: AUC = 0.871374
Fold 3: AUC = 0.775432
Fold 4: AUC = 0.801837
Fold 5: AUC = 0.855827

Mean AUC: 0.822109
Standard Deviation: 0.035807

Standard Deviation (rounded to 3 decimals): 0.036

Answer Q5: 0.036


## Question 6: Hyperparameter Tuning

In [20]:
# C values to test
C_values = [0.000001, 0.001, 1]

results = {}

for C in C_values:
    print(f"\nTesting C = {C}")

    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    c_fold_scores = []

    # Iterate over folds
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(df_full_train)):
        # Split the data
        df_fold_train = df_full_train.iloc[train_idx]
        df_fold_val = df_full_train.iloc[val_idx]

        # Separate features and target
        y_fold_train = df_fold_train['converted'].values
        y_fold_val = df_fold_val['converted'].values

        X_fold_train_df = df_fold_train.drop('converted', axis=1)
        X_fold_val_df = df_fold_val.drop('converted', axis=1)

        # One-hot encode
        train_dict_fold = X_fold_train_df.to_dict(orient='records')
        val_dict_fold = X_fold_val_df.to_dict(orient='records')

        dv_fold = DictVectorizer(sparse=False)
        X_fold_train = dv_fold.fit_transform(train_dict_fold)
        X_fold_val = dv_fold.transform(val_dict_fold)

        # Train model with specific C
        model_fold = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
        model_fold.fit(X_fold_train, y_fold_train)

        # Predict and calculate AUC
        y_fold_pred_proba = model_fold.predict_proba(X_fold_val)[:, 1]
        auc_fold = roc_auc_score(y_fold_val, y_fold_pred_proba)

        c_fold_scores.append(auc_fold)
        print(f"  Fold {fold_idx + 1}: AUC = {auc_fold:.6f}")

    # Calculate mean and std
    c_fold_scores = np.array(c_fold_scores)
    mean_auc_c = np.mean(c_fold_scores)
    std_auc_c = np.std(c_fold_scores)

    # Round to 3 decimals
    mean_auc_c_rounded = round(mean_auc_c, 3)
    std_auc_c_rounded = round(std_auc_c, 3)

    results[C] = {
        'mean': mean_auc_c_rounded,
        'std': std_auc_c_rounded
    }

    print(f"  Mean AUC: {mean_auc_c_rounded}")
    print(f"  Std AUC: {std_auc_c_rounded}")

print()
print("SUMMARY OF HYPERPARAMETER TUNING")


for C in C_values:
    print(f"C = {C}: Mean = {results[C]['mean']}, Std = {results[C]['std']}")

print()

# Find best C (highest mean, if tied then lowest std, if still tied then smallest C)
best_c = None
best_mean = -1
best_std = float('inf')

for C in C_values:
    mean = results[C]['mean']
    std = results[C]['std']

    if mean > best_mean:
        best_c = C
        best_mean = mean
        best_std = std
    elif mean == best_mean and std < best_std:
        best_c = C
        best_std = std
    elif mean == best_mean and std == best_std and C < best_c:
        best_c = C

print(f"Best C: {best_c}")
print(f"Mean: {best_mean}, Std: {best_std}")
print()
print(f"Answer Q6: {best_c}")



Testing C = 1e-06
  Fold 1: AUC = 0.557210
  Fold 2: AUC = 0.519196
  Fold 3: AUC = 0.589491
  Fold 4: AUC = 0.558219
  Fold 5: AUC = 0.576923
  Mean AUC: 0.56
  Std AUC: 0.024

Testing C = 0.001
  Fold 1: AUC = 0.860865
  Fold 2: AUC = 0.896708
  Fold 3: AUC = 0.822816
  Fold 4: AUC = 0.853985
  Fold 5: AUC = 0.900015
  Mean AUC: 0.867
  Std AUC: 0.029

Testing C = 1
  Fold 1: AUC = 0.806075
  Fold 2: AUC = 0.871374
  Fold 3: AUC = 0.775432
  Fold 4: AUC = 0.801837
  Fold 5: AUC = 0.855827
  Mean AUC: 0.822
  Std AUC: 0.036

SUMMARY OF HYPERPARAMETER TUNING
C = 1e-06: Mean = 0.56, Std = 0.024
C = 0.001: Mean = 0.867, Std = 0.029
C = 1: Mean = 0.822, Std = 0.036

Best C: 0.001
Mean: 0.867, Std: 0.029

Answer Q6: 0.001


##Summary of all the answers:

In [22]:
print("Summary Results:")
print()
print("Q1 - ROC AUC feature importance: number_of_courses_viewed")
print("Q2 - Model AUC on validation: 0.817")
print("Q3 - Precision-Recall intersection: 0.545")
print("Q4 - Maximum F1 threshold: 0.54")
print("Q5 - Std of 5-Fold CV scores: 0.036")
print("Q6 - Best C parameter: 0.001")

Summary Results:

Q1 - ROC AUC feature importance: number_of_courses_viewed
Q2 - Model AUC on validation: 0.817
Q3 - Precision-Recall intersection: 0.545
Q4 - Maximum F1 threshold: 0.54
Q5 - Std of 5-Fold CV scores: 0.036
Q6 - Best C parameter: 0.001
